In [71]:
import pandas as pd
import numpy as np
import ast
import plotly.express as px

In [72]:
df_sales_2025 = pd.read_csv('vg_sales_2025.csv', index_col=False, parse_dates=["release_date", "last_update"])
pd.set_option('display.max_rows', None)

In [73]:
# Limpieza - datos nulos, duplicados, y preparación


# 0 - Creación de funciones y eliminación de columnas innecesarias y duplicados.

def null_values_fields(dataframe):
    null_counts = dataframe.isna().sum()
    return null_counts[null_counts > 0]

def sales_null_filler(total_sales, sales):
    to_fill = sales.pop(0)
    to_fill_clean = to_fill.fillna(
        total_sales
        - sum(x.fillna(0) for x in sales)
    )
    sales.append(to_fill_clean)
    return sales, to_fill_clean


# Dropeamos columna que no podemos utilizar para ningún tipo de análisis. 
# 'img' no nos sirve, no tenemos acceso a las imágenes. 'last_update' podría habernos dado información interesante para el análisis, pero el 77% de los datos útiles es faltante en esa columna.
df_sales_2025 = df_sales_2025.drop(columns=['img', 'last_update'])

df_sales_2025 = df_sales_2025.drop_duplicates(subset=['title', 'console'])


# 1 - Manejo de nulos de columnas de ventas

total_sales_null = df_sales_2025[df_sales_2025["total_sales"].isna()]
print(f"1. Nulos con venta en nulo: \n{null_values_fields(total_sales_null)}")
df_sales_2025 = df_sales_2025.dropna(subset=['total_shipped', 'total_sales'], how='all')

sales = [df_sales_2025['other_sales'], df_sales_2025['pal_sales'], df_sales_2025['na_sales'], df_sales_2025['jp_sales']]
df_sales_2025 = df_sales_2025[(df_sales_2025['total_sales'] > 0) | (df_sales_2025['total_shipped'] > 0)]

print(f"2. Nulos de dataset después del paso 0: \n{null_values_fields(df_sales_2025)}")

# Utilizamos la función sales_null_filler para rellenar los valores nulos de cada columna de ventas sin asumir que valen 0.

# df_sales_2025 = df_sales_2025.sort_values(by=['total_shipped', 'total_sales'], ascending=False)

df_notna = df_sales_2025[df_sales_2025['total_sales'].notna()].copy()
total_sales_notna = df_notna['total_sales']

sales, df_notna['other_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['pal_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['na_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)
sales, df_notna['jp_sales'] = sales_null_filler(
    total_sales_notna,
    sales
)

df_sales_2025.loc[df_notna.index, ["other_sales", "pal_sales", "na_sales", "jp_sales"]] = \
    df_notna[["other_sales", "pal_sales", "na_sales", "jp_sales"]]

print(f"3. Nulos de dataset después del paso 1: \n{null_values_fields(df_sales_2025)}")


# 2 - Manejo de nulos de las columnas developer, release_date y critic_score


# Al ser pocos valores nulos en la columna 'developer', se hizo una búsqueda para encontrar los valores faltantes, utilizando como fuente páginas oficiales de documentación de los juegos.
corrections = {
    'Gourmet Chef: Cook Your Way to Fame': 'Creative Patterns',
    'Wordmaster': 'Sarbakan'
}
df_sales_2025["developer"] = df_sales_2025['developer'].fillna(
    df_sales_2025['title'].map(corrections)
)


with open("date_corrections.txt", "r", encoding="utf-8") as file:
    date_corrections = file.read()

date_corrections_dict = ast.literal_eval(date_corrections)

df_sales_2025['release_date'] = df_sales_2025['release_date'].fillna(
    df_sales_2025['title'].map(date_corrections_dict)
)

print(f"4. Nulos de dataset después del paso 2: \n{null_values_fields(df_sales_2025)}")

# Después de investigar acerca de las filas que tenían release_date nulo, se descubrió que un 75% de las filas correspondían a juegos de PC.

pc = df_sales_2025["console"] == "PC"

print(df_sales_2025.loc[pc, "release_date"].isna().mean())

# Debido a que un 17% de los valores de release_date de los juegos de PC están vacíos, se decidió no eliminar las filas vacías de fecha por ahora, y en cambio hacerlo solo si necesitamos un análisis de tiempo.

# Finalmente, creamos una columna que contenga el año de salida de cada juego en lugar de la fecha completa.

df_sales_2025['year'] = df_sales_2025['release_date'].dt.year
df_sales_2025['year'] = df_sales_2025['year'].astype('Int64')


1. Nulos con venta en nulo: 
developer           13
vg_score         46374
critic_score     45432
user_score       47822
total_shipped    43425
total_sales      48078
na_sales         48078
jp_sales         48078
pal_sales        48078
other_sales      48078
release_date      9251
dtype: int64
2. Nulos de dataset después del paso 0: 
developer            2
vg_score         21005
critic_score     17225
user_score       21837
total_shipped    17485
total_sales       4637
na_sales          9699
jp_sales         15703
pal_sales        10379
other_sales       8021
release_date       531
dtype: int64
3. Nulos de dataset después del paso 1: 
developer            2
vg_score         21005
critic_score     17225
user_score       21837
total_shipped    17485
total_sales       4637
na_sales          4637
jp_sales          4637
pal_sales         4637
other_sales       4637
release_date       531
dtype: int64
4. Nulos de dataset después del paso 2: 
vg_score         21005
critic_score     17225
user

In [ ]:
# Desde aquí, se hará una separación del dataset en 2. Una parte tendrá todos los registros que tengan 'Series' en su campo console, ya que representan sagas completas de juegos.
# El otro dataset será el de los juegos individuales o 'All' (que representan al juego en todas sus plataformas en lugar de una específica)

df_sales_series = df_sales_2025[df_sales_2025['console'] == 'Series'].copy()
df_sales_ind = df_sales_2025[~df_sales_2025.index.isin(df_sales_series.index)].copy()

print(f"4. Nulos de dataset individual: \n{null_values_fields(df_sales_ind)}")
print(f"4. Nulos de dataset series: \n{null_values_fields(df_sales_series)}")

# Debido a que el dataset Series tiene varias columnas completamente vacías, se eliminarán dichas columnas

df_sales_series = df_sales_series.dropna(axis=1, how='all')

print(df_sales_ind["user_score"].isna().mean())
print(df_sales_ind["critic_score"].isna().mean())
print(df_sales_ind["vg_score"].isna().mean())

df_sales_ind = df_sales_ind.drop(columns=['vg_score'])

# Debido a la alta proporción de datos faltantes de las tres columnas de puntuación, se decidió eliminar la columna del puntaje de vgchartz, pero dejar las otras dos por si se quieren analizar.


df_sales_ind["units_reported"] = (
    df_sales_ind["total_sales"]
    .fillna(df_sales_ind["total_shipped"])
)

df_sales_ind = df_sales_ind.sort_values('units_reported', ascending=False)
df_sales_series = df_sales_series.sort_values('total_shipped', ascending=False)

print(df_sales_ind['console'].value_counts(normalize=True))

# Considerar para más adelante que se pueden quitar los juegos All y hacer un análisis con solo juegos individuales, o por el contrario eliminar duplicados por consola y quedarnos con los All y versiones con más ventas de los juegos.


# 3 - Manejo de valores inconsistentes


print(df_sales_ind['console'].unique())
display(df_sales_ind[df_sales_2025['console'] == 'WW'])

# Al investigar, se descubre que el único registro que tiene valor 'WW' en el campo 'console' es en realidad un juego de la Wii, por lo que se corrige.

df_sales_ind.loc[6132, 'console'] = 'Wii'

# Después de revisar el resto de columnas, no se encontraron más valores inconsistentes.

print(df_sales_ind.info())
print(df_sales_series.info())

# El dataset finaliza la limpeza con 22.123 filas restantes de 67.172 originales. Aunque solo quedan el 33,3% de las filas originales, 
# Más de 22000 filas siguen siendo datos suficientes para el análisis que se desea realizar.

4. Nulos de dataset individual: 
vg_score         20522
critic_score     16742
user_score       21354
total_shipped    17485
total_sales       4154
na_sales          4154
jp_sales          4154
pal_sales         4154
other_sales       4154
release_date       492
year               492
dtype: int64
4. Nulos de dataset series: 
vg_score        483
critic_score    483
user_score      483
total_sales     483
na_sales        483
jp_sales        483
pal_sales       483
other_sales     483
dtype: int64
0.9868293359212533
0.7736956421276399
0.9483802393825962
console
PC      0.120292
DS      0.104164
PS2     0.102870
PS3     0.063265
Wii     0.062803
PSP     0.060816
X360    0.060308
PS      0.055686
All     0.055455
PS4     0.043995
GBA     0.039050
XB      0.038542
PSV     0.029530
3DS     0.028467
GC      0.026018
XOne    0.025047
NS      0.017191
N64     0.014742
SNES    0.011184
SAT     0.008133
WiiU    0.007394
2600    0.006146
NES     0.005083
GB      0.004298
DC      0.002449
GEN     0

C:\Users\dschu\AppData\Local\Temp\ipykernel_2612\2442294710.py:38: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  display(df_sales_ind[df_sales_2025['console'] == 'WW'])


,title,console,genre,publisher,developer,vg_score,critic_score,user_score,total_shipped,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,year,units_reported
21044,Karaoke Joysound Wii,WW,Misc,Hudson Soft,Xing Inc.,NaN,NaN,NaN,NaN,0.25,0.0,0.25,0.0,0.0,2009-07-29,2009,0.25


<class 'pandas.core.frame.DataFrame'>
Index: 21640 entries, 15594 to 6132
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           21639 non-null  object        
 1   console         21640 non-null  object        
 2   genre           21639 non-null  object        
 3   publisher       21639 non-null  object        
 4   developer       21639 non-null  object        
 5   vg_score        1117 non-null   float64       
 6   critic_score    4897 non-null   float64       
 7   user_score      285 non-null    float64       
 8   total_shipped   4154 non-null   float64       
 9   total_sales     17485 non-null  float64       
 10  na_sales        17485 non-null  float64       
 11  jp_sales        17485 non-null  float64       
 12  pal_sales       17485 non-null  float64       
 13  other_sales     17485 non-null  float64       
 14  release_date    21147 non-null  datetime64[ns]
 15  year

In [75]:
df_sales_ind.to_csv('sales_ind_clean.csv', index=False)
df_sales_series.to_csv('sales_series_clean.csv', index=False)

In [ ]:

# df_sales_2024 = df_sales_2025.dropna(subset=['release_date'])
